# Thí nghiệm Hybrid 1D-CNN + LSTM

Notebook này chạy lại pipeline trong `CNN_LSTM.py`.

In [1]:
from pathlib import Path
import os

START_DIR = Path.cwd().resolve()
if (START_DIR / 'cnn_lstm' / 'CNN_LSTM.py').exists():
    PROJECT_ROOT = START_DIR
    NOTEBOOK_DIR = PROJECT_ROOT / 'cnn_lstm'
elif (START_DIR / 'CNN_LSTM.py').exists() and START_DIR.name == 'cnn_lstm':
    NOTEBOOK_DIR = START_DIR
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    raise FileNotFoundError('Run this notebook from the repo root or the cnn_lstm directory.')

os.chdir(NOTEBOOK_DIR)

required = [
    NOTEBOOK_DIR / 'CNN_LSTM.py',
    PROJECT_ROOT / 'SQLInjection_XSS_MixDataset.1.0.0.csv',
    PROJECT_ROOT / 'csic_database.csv',
    PROJECT_ROOT / 'obfuscation_dataset_full.xlsx',
]
for item in required:
    assert item.exists(), f'Missing file: {item}'

print('Start dir:', START_DIR)
print('Notebook dir:', NOTEBOOK_DIR)
print('Project root:', PROJECT_ROOT)
print('Found CNN_LSTM.py and all datasets.')


Start dir: C:\StudyWithKhang\NCKH\obfuscated-web-attack-detection\cnn_lstm
Notebook dir: C:\StudyWithKhang\NCKH\obfuscated-web-attack-detection\cnn_lstm
Project root: C:\StudyWithKhang\NCKH\obfuscated-web-attack-detection
Found CNN_LSTM.py and all datasets.


## 1. Kiểm tra kiến trúc
Cell này chỉ dựng model và in shape, không huấn luyện.

In [2]:
import importlib.util
import sys

module_path = NOTEBOOK_DIR / 'CNN_LSTM.py'
spec = importlib.util.spec_from_file_location('nckh_test_pipeline', module_path)
pipeline = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = pipeline
spec.loader.exec_module(pipeline)

demo_model = pipeline.build_model(vocab_size=191, max_len=1024, embedding_dim=64)
demo_model.summary()


Model: "Hybrid_1D_CNN_LSTM_Web_Attack_Detector"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ char_embedding (Embedding)      │ (None, 1024, 64)       │        12,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_k3 (Conv1D)                │ (None, 1024, 128)      │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool_1 (MaxPooling1D)           │ (None, 256, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_k5 (Conv1D)                │ (None, 256, 128)       │        82,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool_2 (MaxPooling1D)           │ (None, 64, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_context (LSTM)             │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_classifier (Dense)        │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ attack_probability (Dense)      │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 258,881 (1011.25 KB)

 Trainable params: 258,881 (1011.25 KB)

 Non-trainable params: 0 (0.00 B)

## 2. Smoke test tùy chọn
Chạy trên mẫu nhỏ để kiểm tra môi trường. Không dùng kết quả này trong báo cáo.

In [3]:
%run CNN_LSTM.py --sample-size 3000 --obfu-sample-size 1000 --epochs 3 --batch-size 128 --output-dir artifacts_cnn_lstm_smoke

=== DATA PREPARED ===
Train           : 2,160
Val             : 240
Test            : 600
Obfuscated test : 1,000
Base p99 length : 981

=== TOKENIZER ===
Vocabulary size: 119
Tokenizer was fit on train payloads only.

=== INPUT SHAPES ===
X_train: (2160, 1024)
X_val  : (240, 1024)
X_test : (600, 1024)
X_obfu : (1000, 1024)
Class weights: {0: 1.4457831325301205, 1: 0.7643312101910829}


Model: "Hybrid_1D_CNN_LSTM_Web_Attack_Detector"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ char_embedding (Embedding)      │ (None, 1024, 64)       │         7,616 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_k3 (Conv1D)                │ (None, 1024, 128)      │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool_1 (MaxPooling1D)           │ (None, 256, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_k5 (Conv1D)                │ (None, 256, 128)       │        82,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool_2 (MaxPooling1D)           │ (None, 64, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_context (LSTM)             │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_classifier (Dense)        │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ attack_probability (Dense)      │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 254,273 (993.25 KB)

 Trainable params: 254,273 (993.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/3
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 297ms/step - accuracy: 0.5140 - auc: 0.5449 - loss: 0.6886 - precision: 0.6777 - recall: 0.4908
Epoch 1: val_loss improved from None to 0.52645, saving model to artifacts_cnn_lstm_smoke\best_hybrid_cnn_lstm.keras

Epoch 1: finished saving model to artifacts_cnn_lstm_smoke\best_hybrid_cnn_lstm.keras
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 345ms/step - accuracy: 0.5454 - auc: 0.6513 - loss: 0.6592 - precision: 0.7515 - recall: 0.4558 - val_accuracy: 0.8292 - val_auc: 0.8904 - val_loss: 0.5264 - val_precision: 0.8919 - val_recall: 0.8408
Epoch 2/3
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 269ms/step - accuracy: 0.8400 - auc: 0.8356 - loss: 0.4630 - precision: 0.8846 - recall: 0.8677
Epoch 2: val_loss improved from 0.52645 to 0.43737, saving model to artifacts_cnn_lstm_smoke\best_hybrid_cnn_lstm.keras

Epoch 2: finished saving model to artifacts_cnn_lstm_smoke\best_hybrid_cnn_lstm.keras
17/17 ━━━━━━━━━━━━━━━━━━━━ 5s 287ms/step - accuracy: 0.8333 - auc: 0.8306 - loss: 0.47

c:\StudyWithKhang\NCKH\obfuscated-web-attack-detection\myvenv\Lib\site-packages\sklearn\metrics\_classification.py:614: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


## 3. Huấn luyện đầy đủ
Cell này chạy toàn bộ dữ liệu. EarlyStopping có thể dừng trước 50 epoch. Thời gian chạy trên CPU có thể kéo dài nhiều giờ.

In [4]:
%run CNN_LSTM.py --epochs 50 --batch-size 128 --output-dir artifacts_cnn_lstm_rerun

=== DATA PREPARED ===
Train           : 117,444
Val             : 13,050
Test            : 32,624
Obfuscated test : 150,000
Base p99 length : 977

=== TOKENIZER ===
Vocabulary size: 180
Tokenizer was fit on train payloads only.

=== INPUT SHAPES ===
X_train: (117444, 1024)
X_val  : (13050, 1024)
X_test : (32624, 1024)
X_obfu : (150000, 1024)
Class weights: {0: 1.3983093225383973, 1: 0.7783005738975997}


Model: "Hybrid_1D_CNN_LSTM_Web_Attack_Detector"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ char_embedding (Embedding)      │ (None, 1024, 64)       │        11,520 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_k3 (Conv1D)                │ (None, 1024, 128)      │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool_1 (MaxPooling1D)           │ (None, 256, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_k5 (Conv1D)                │ (None, 256, 128)       │        82,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool_2 (MaxPooling1D)           │ (None, 64, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_context (LSTM)             │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_classifier (Dense)        │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ attack_probability (Dense)      │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 258,177 (1008.50 KB)

 Trainable params: 258,177 (1008.50 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
918/918 ━━━━━━━━━━━━━━━━━━━━ 0s 261ms/step - accuracy: 0.8418 - auc: 0.8767 - loss: 0.4165 - precision: 0.8771 - recall: 0.8799
Epoch 1: val_loss improved from None to 0.08848, saving model to artifacts_cnn_lstm_rerun\best_hybrid_cnn_lstm.keras

Epoch 1: finished saving model to artifacts_cnn_lstm_rerun\best_hybrid_cnn_lstm.keras
918/918 ━━━━━━━━━━━━━━━━━━━━ 251s 270ms/step - accuracy: 0.8868 - auc: 0.9384 - loss: 0.3094 - precision: 0.9172 - recall: 0.9054 - val_accuracy: 0.9542 - val_auc: 0.9941 - val_loss: 0.0885 - val_precision: 0.9938 - val_recall: 0.9345
Epoch 2/50
918/918 ━━━━━━━━━━━━━━━━━━━━ 0s 274ms/step - accuracy: 0.9650 - auc: 0.9937 - loss: 0.0850 - precision: 0.9887 - recall: 0.9564
Epoch 2: val_loss improved from 0.08848 to 0.05823, saving model to artifacts_cnn_lstm_rerun\best_hybrid_cnn_lstm.keras

Epoch 2: finished saving model to artifacts_cnn_lstm_rerun\best_hybrid_cnn_lstm.keras
918/918 ━━━━━━━━━━━━━━━━━━━━ 261s 284ms/step - accuracy: 0.9699 - auc: 0.995

## 4. Đọc kết quả CNN-LSTM vừa chạy

In [ ]:
import json

candidate_result_dirs = [
    NOTEBOOK_DIR / 'artifacts_cnn_lstm_rerun',
    NOTEBOOK_DIR / 'artifacts',
    NOTEBOOK_DIR / 'artifacts_cnn_lstm_smoke',
]
result_dir = next(
    (item for item in candidate_result_dirs if (item / 'metadata_and_results.json').exists()),
    None,
)
if result_dir is None:
    checked = '\n'.join(str(item) for item in candidate_result_dirs)
    raise FileNotFoundError('No CNN-LSTM metadata_and_results.json found. Checked:\n' + checked)

print('Reading CNN-LSTM results from:', result_dir)
with (result_dir / 'metadata_and_results.json').open(encoding='utf-8') as f:
    cnn_lstm_results = json.load(f)

cnn_lstm_results['model'], cnn_lstm_results['evaluation']


## 5. So sánh CNN-only và CNN-LSTM
Hai model dùng cùng dữ liệu, tokenizer, seed, class weight và threshold 0.5.

In [ ]:
import pandas as pd

cnn_only_results_path = PROJECT_ROOT / 'cnn_only' / 'artifacts' / 'metadata_and_results.json'
if not cnn_only_results_path.exists():
    raise FileNotFoundError(
        'CNN-only results not found. Run cnn_only first or skip this comparison cell: '
        + str(cnn_only_results_path)
    )

with cnn_only_results_path.open(encoding='utf-8') as f:
    cnn_results = json.load(f)

cnn_eval = cnn_results['evaluation']
hybrid_eval = cnn_lstm_results['evaluation']
cnn_attack = cnn_eval['normal_test']['classification_report']['Attack (1)']
hybrid_attack = hybrid_eval['test']['classification_report']['Attack (1)']

comparison = pd.DataFrame([
    {
        'Model': 'CNN-only',
        'Params': cnn_results['cnn_only_model']['parameter_count'],
        'Test Accuracy': cnn_eval['normal_test']['accuracy'],
        'Test AUC': cnn_eval['normal_test']['auc_roc'],
        'Attack Recall': cnn_attack['recall'],
        'Attack F1': cnn_attack['f1-score'],
        'Obfuscated Recall': cnn_eval['obfuscated_test']['classification_report']['1']['recall'],
        'Obfuscated Missed': cnn_eval['obfuscated_test']['confusion_matrix'][1][0],
    },
    {
        'Model': 'CNN-LSTM rerun',
        'Params': None,
        'Test Accuracy': hybrid_eval['test']['accuracy'],
        'Test AUC': hybrid_eval['test']['auc_roc'],
        'Attack Recall': hybrid_attack['recall'],
        'Attack F1': hybrid_attack['f1-score'],
        'Obfuscated Recall': hybrid_eval['obfuscated_test']['classification_report']['1']['recall'],
        'Obfuscated Missed': hybrid_eval['obfuscated_test']['confusion_matrix'][1][0],
    },
])
comparison


## Nguyên tắc kết luận

- Nếu CNN-LSTM tăng rõ Attack Recall hoặc Obfuscated Recall, LSTM có đóng góp thực nghiệm.
- Nếu kết quả gần ngang, ưu tiên CNN-only vì nhẹ hơn.
- Không kết luận chỉ dựa trên accuracy; cần xem False Negative, recall obfuscation, số tham số và thời gian chạy.